In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import medfilt, savgol_filter
import pybaselines

# ---- Config ----
LASER_WAVELENGTH_NM = 785.0
N_POINTS = 1024
N_REPS = 300

data_dir = Path("/Users/yifeigu/Documents/Carney_Lab/RamanPy/data/2026_03_05_1/capture")
spec_files = sorted((data_dir / "spectra").glob("*.txt"))
print(f"Found {len(spec_files)} spectrum files: {[f.name for f in spec_files]}")

In [ ]:
# ---- Helper functions ----
def wavelength_to_raman_shift(wavelength_nm, laser_nm=785.0):
    return (1.0 / laser_nm - 1.0 / wavelength_nm) * 1e7

def remove_cosmic_rays(intensity, threshold=10):
    median_filtered = medfilt(intensity, kernel_size=5)
    residual = intensity - median_filtered
    std_dev = np.std(residual)
    mask = np.abs(residual) > (threshold * std_dev)
    result = intensity.copy()
    result[mask] = median_filtered[mask]
    return result

def airpls_baseline(intensity, raman_shifts, lam=100, diff_order=1, max_iter=15, tol=0.005):
    fitter = pybaselines.Baseline(x_data=raman_shifts)
    baseline, _ = fitter.airpls(intensity, lam, diff_order, max_iter, tol)
    return intensity - baseline

def normalize_spectrum(intensity):
    return (intensity - np.min(intensity)) / (np.max(intensity) - np.min(intensity))

def preprocess_spectrum(intensity, raman_shifts):
    """Cosmic ray removal -> Crop -> Baseline -> Smooth -> Normalize"""
    result = remove_cosmic_rays(intensity, threshold=10)
    # Crop to fingerprint region
    crop_mask = (raman_shifts >= 662.697) & (raman_shifts <= 1784.104)
    rs = raman_shifts[crop_mask]
    result = result[crop_mask]
    # Baseline correction
    result = airpls_baseline(result, rs)
    # Smoothing
    result = savgol_filter(result, 5, 3)
    # Normalization
    result = normalize_spectrum(result)
    return result, rs

In [ ]:
# ---- Load all spectra ----
all_raw = {}  # {filename: (raman_shifts, spectra_array[300, 1024])}

for fpath in spec_files:
    data = np.loadtxt(fpath, delimiter=',')
    wavelengths = data[:N_POINTS, 0]
    raman_shifts = wavelength_to_raman_shift(wavelengths, LASER_WAVELENGTH_NM)
    intensities = data[:, 1].reshape(N_REPS, N_POINTS)
    all_raw[fpath.stem] = (raman_shifts, intensities)
    print(f"{fpath.name}: {intensities.shape[0]} spectra, {intensities.shape[1]} points")
    print(f"  Raman shift range: {raman_shifts.min():.1f} - {raman_shifts.max():.1f} cm^-1")

In [ ]:
# ---- Plot RAW spectra colored by sequence (turbo colormap) ----
plt.style.use('dark_background')
fig, axes = plt.subplots(1, 2, figsize=(20, 7), dpi=150)
fig.patch.set_facecolor('black')

for ax, (name, (rs, spectra)) in zip(axes, all_raw.items()):
    ax.set_facecolor('black')
    colors = plt.cm.turbo(np.linspace(0, 1, N_REPS))
    for i in range(N_REPS):
        ax.plot(rs, spectra[i], color=colors[i], linewidth=0.3, alpha=0.6)
    ax.set_xlabel('Raman Shift (cm$^{-1}$)', fontsize=14)
    ax.set_ylabel('Intensity (counts)', fontsize=14)
    ax.set_title(f'{name} — Raw (n={N_REPS})', fontsize=15)

# Add shared colorbar
sm = plt.cm.ScalarMappable(cmap='turbo', norm=plt.Normalize(vmin=1, vmax=N_REPS))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, shrink=0.85, pad=0.02)
cbar.set_label('Spectrum # (sequence)', fontsize=13)

plt.tight_layout()
plt.savefig(data_dir / 'raw_spectra_turbo.png', dpi=150, facecolor='black', bbox_inches='tight')
plt.show()

In [ ]:
# ---- Preprocess all spectra ----
all_processed = {}

for name, (rs, spectra) in all_raw.items():
    processed_list = []
    for i in range(N_REPS):
        proc_int, proc_rs = preprocess_spectrum(spectra[i], rs)
        processed_list.append(proc_int)
    processed_arr = np.array(processed_list)
    all_processed[name] = (proc_rs, processed_arr)
    print(f"{name}: preprocessed shape {processed_arr.shape}")

In [ ]:
# ---- Plot PROCESSED spectra colored by sequence (turbo colormap) ----
fig, axes = plt.subplots(1, 2, figsize=(20, 7), dpi=150)
fig.patch.set_facecolor('black')

for ax, (name, (rs, spectra)) in zip(axes, all_processed.items()):
    ax.set_facecolor('black')
    colors = plt.cm.turbo(np.linspace(0, 1, N_REPS))
    for i in range(N_REPS):
        ax.plot(rs, spectra[i], color=colors[i], linewidth=0.3, alpha=0.6)
    ax.set_xlabel('Raman Shift (cm$^{-1}$)', fontsize=14)
    ax.set_ylabel('Intensity (a.u.)', fontsize=14)
    ax.set_title(f'{name} — Processed (n={N_REPS})', fontsize=15)
    ax.set_ylim(-0.15, 1.1)

# Add shared colorbar
sm = plt.cm.ScalarMappable(cmap='turbo', norm=plt.Normalize(vmin=1, vmax=N_REPS))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, shrink=0.85, pad=0.02)
cbar.set_label('Spectrum # (sequence)', fontsize=13)

plt.tight_layout()
plt.savefig(data_dir / 'processed_spectra_turbo.png', dpi=150, facecolor='black', bbox_inches='tight')
plt.show()